In [ ]:
import json

import pandas as pd
import tensorflow as tf
from keras import Model
from keras.layers import TextVectorization
from keras.models import load_model

from models.loss import loss_that_ignores_padding
from utils.load_image import load_image

In [ ]:
model: Model = load_model(
    "../models/model_train.keras",
    custom_objects={
        "loss_that_ignores_padding": loss_that_ignores_padding
    }
)

test_df = pd.read_csv("../flickr8k/captions_test.csv")

test_image = test_df["image"].to_list()[0]
test_caption = test_df["caption"].to_list()[0]

print(test_image)

image_path = f"../flickr8k/Images/{test_image}"

with open("../config/config.json", "r") as f:
    config = json.load(f)

vectorizer = TextVectorization.from_config(
    config["vec_config"]
)

max_len = config["max_len"]

In [ ]:
def generate_caption(
        image_path: str,
        model,
        vectorizer,
        max_len: int
):
    img = load_image(image_path)
    img = tf.expand_dims(img, axis=0)

    vocab = vectorizer.get_vocabulary()
    id_to_word = dict(enumerate(vocab))

    caption = "[START]"

    for _ in range(max_len):

        seq = vectorizer([caption])

        preds = model.predict(
            (img, seq),
            verbose=0
        )

        current_length = len(caption.split())

        next_token_id = tf.argmax(
            preds[0, current_length - 1]
        ).numpy()

        next_word = id_to_word[next_token_id]

        if next_word == "[END]":
            break

        caption += " " + next_word

    caption = (
        caption
        .replace("[START]", "")
        .replace("[END]", "")
        .strip()
    )

    return caption

In [ ]:
caption = generate_caption(image_path, model, vectorizer, max_len)

print(f"Generated caption: {caption}")
print(f"Real caption: {test_caption}")